In [1]:
!pip install requests pandas scikit-learn

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(url, sep="\t", header=None)
df.columns = ["label", "text"]

df["label"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
import requests
import json

def ask_llm(prompt):
    url = "http://localhost:11434/api/generate"

    data = {
        "model": "qwen2.5:0.5b",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=data)
    result = response.json()

    return result.get("response", "")

In [4]:
def build_prompt(text):
    return f"""
You are a spam classifier.

Return ONLY JSON:

{{"reasoning": "...", "verdict": 0}}

Text: {text}
"""

In [5]:
sample = df.iloc[0]["text"]

response = ask_llm(build_prompt(sample))
print(response)

{"reasoning": "The given text does not contain any spam elements or patterns that would indicate it's from a spam email. It appears to be a standard customer service query related to a specific location and event.", "verdict": 0}


In [6]:
def extract_verdict(text: str) -> int:
    """
    Извлекает бинарный ответ из вывода модели.
    Возвращает:
        1 — spam
        0 — not spam
    """
    text = text.lower()

    if "spam" in text:
        return 1
    elif "not spam" in text or "ham" in text:
        return 0
    else:
        return 0  # fallback

In [7]:
extract_verdict(response)

1

In [8]:
results = []

for i in range(10):
    text = df.iloc[i]["text"]
    label = df.iloc[i]["label"]

    out = ask_llm(build_prompt(text))

    results.append((text, label, out))

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = []
y_pred = []

for text, label, out in results:
    y_true.append(label)
    y_pred.append(extract_verdict(out))

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

Accuracy: 0.7
Precision: 0.5714285714285714
Recall: 1.0
F1: 0.7272727272727273


In [16]:
prompts = [
    "Hello, how are you?",
    "Win a free iPhone now!!!",
    "Meeting at 5pm",
    "Cheap loans available",
    "Call me later",
    "Congratulations! You won money",
    "Let's go for dinner",
    "Limited offer!!! Buy now",
    "Are you coming today?",
    "Click this link to claim prize"
]

simple_results = []

for p in prompts:
    response = ask_llm(p)
    simple_results.append((p, response))

In [17]:
import pandas as pd

prompts_list = []
responses_list = []

for text, label, out in results:
    prompts_list.append(text)
    responses_list.append(out)

simple_report = pd.DataFrame({
    "prompt": prompts_list,
    "response": responses_list
})

simple_report.to_csv("llm_report.csv", index=False)